In [1]:
from google.cloud import bigquery

PROJECT_ID = "qwiklabs-gcp-00-f469c11ef7ce"
DATASET = "weather_comms"
SOURCE_URI = "gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv"
CONNECTION = "gemini_conn"

client = bigquery.Client(project=PROJECT_ID, location="US")

connected


In [2]:
RAW_TABLE = f"{PROJECT_ID}.{DATASET}.weather_raw"

client.query(f"CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}` OPTIONS(location='US')").result()

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
client.load_table_from_uri(SOURCE_URI, RAW_TABLE, job_config=job_config).result()

table = client.get_table(RAW_TABLE)
print(f"{table.num_rows:,} rows, {len(table.schema)} columns")

300 rows, 9 columns


In [3]:
for f in table.schema:
    print(f"{f.name:30s} {f.field_type}")
print("-" * 50)
client.query(f"SELECT * FROM `{RAW_TABLE}` LIMIT 5").to_dataframe()

date                           DATE
city                           STRING
state                          STRING
temperature_f                  FLOAT
wind_speed_mph                 FLOAT
precipitation_in               FLOAT
barometric_pressure_inHg       FLOAT
humidity_percent               FLOAT
weather_condition              STRING
--------------------------------------------------


,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy


In [5]:
client.query(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.gemini_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.us.{CONNECTION}`
OPTIONS (endpoint = 'gemini-2.5-flash')
""").result()
print("Gemini model created")

Gemini model created


In [6]:
generate_sql = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.weather_reports` AS
SELECT
  date, city, state, weather_condition, temperature_f,
  wind_speed_mph, precipitation_in, humidity_percent,
  ml_generate_text_llm_result AS weather_advisory
FROM ML.GENERATE_TEXT(
  MODEL `{PROJECT_ID}.{DATASET}.gemini_model`,
  (
    SELECT
      *,
      CONCAT(
        'You are a public safety communications officer for a city emergency management office. ',
        'Write a brief, clear weather advisory (2-3 sentences) for residents in plain language, no jargon. ',
        'If conditions are hazardous (extreme heat or cold, high winds, heavy rain), name the hazard and give one practical safety tip. ',
        'If conditions are mild, give a short reassuring note. Do not invent data beyond what is provided.\\n\\n',
        'Location: ', city, ', ', state, '\\n',
        'Date: ', CAST(date AS STRING), '\\n',
        'Condition: ', weather_condition, '\\n',
        'Temperature: ', CAST(temperature_f AS STRING), ' F\\n',
        'Wind: ', CAST(wind_speed_mph AS STRING), ' mph\\n',
        'Precipitation: ', CAST(precipitation_in AS STRING), ' in\\n',
        'Humidity: ', CAST(humidity_percent AS STRING), '%'
      ) AS prompt
    FROM `{RAW_TABLE}`
  ),
  STRUCT(
    0.3 AS temperature,
    512 AS max_output_tokens,
    TRUE AS flatten_json_output
  )
)
"""
client.query(generate_sql).result()
print("Advisories generated and saved -> weather_reports")

Advisories generated and saved -> weather_reports


In [7]:
client.query(f"""
SELECT city, state, weather_condition, temperature_f, weather_advisory
FROM `{PROJECT_ID}.{DATASET}.weather_reports`
LIMIT 5
""").to_dataframe()

,city,state,weather_condition,temperature_f,weather_advisory
0,Atlanta,GA,Cloudy,59.5,"Good morning, Atlanta! Expect a cloudy day wit..."
1,Atlanta,GA,Cloudy,51.7,"Good morning, Atlanta! Expect a cloudy day wit..."
2,Atlanta,GA,Cloudy,75.2,"Atlanta, expect a cloudy day with comfortable ..."
3,Atlanta,GA,Cloudy,71.7,"Atlanta, expect a cloudy day with mild tempera..."
4,Atlanta,GA,Cloudy,74.4,"Good morning, Atlanta! Expect a cloudy day wit..."
